# Seminar: Page-CUSUM for a mean shift

In this variant you will build a small change-point detector end to end.

**What you need to do**
1. Generate a time series whose mean stays constant and then jumps.
2. Derive and implement **Page-CUSUM** for a Gaussian mean shift.
3. Measure performance with two numbers:
   - **detection delay** — how late you react after the true change;
   - **time to false alarm (TFA)** — how long the detector survives without a false alarm on a series *with no change*.

### The Page-CUSUM idea

At each time $t$ we compare two hypotheses for the observation $x_t$:

- $H_0$: density $f_0$ (pre-change regime);
- $H_1$: density $f_1$ (post-change regime).

We accumulate the log-likelihood ratio, but never let the statistic go below zero:

$$
S_t = \max\bigl(0,\; S_{t-1} + \ell_t\bigr),
\quad
\ell_t = \log \frac{f_1(x_t)}{f_0(x_t)},
\quad
S_0 = 0.
$$

An alarm is raised when $S_t$ crosses a threshold $h$:

$$
\tau = \min\{ t : S_t > h \}.
$$

Your job later is to **derive** what $\ell_t$ becomes in the Gaussian case — we will not hand you the simplified formula.

## 0. Imports and experiment parameters

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# How the series is generated
MU0 = 0.0          # mean BEFORE the change
MU1 = 1.0          # mean AFTER the change
SIGMA = 1.0        # noise level
THETA = 200        # true change time
N = 400            # length of the series with a change
N_FA = 2000        # length of the no-change series (for TFA)
SEED = 42

# Detector parameter
H = 5.0   # alarm threshold — worth playing with

rng = np.random.default_rng(SEED)
print(f"mu0 = {MU0}, mu1 = {MU1}, sigma = {SIGMA}, h = {H}")


## 1. Generate a series with a change point

We use the simplest useful scenario: before time $\theta$ the points fluctuate around $\mu_0$, afterwards around $\mu_1$.

- for $t < \theta$: $x_t \sim \mathcal{N}(\mu_0, \sigma^2)$
- for $t \ge \theta$: $x_t \sim \mathcal{N}(\mu_1, \sigma^2)$

Fill in the TODOs below, then check the plot: you should be able to see the change by eye.

In [ ]:
def generate_mean_shift(
    n,
    theta,
    mu0=MU0,
    mu1=MU1,
    sigma=SIGMA,
    rng=None,
):
    """Gaussian series with a mean shift at time theta."""
    rng = rng or np.random.default_rng()
    x = # TODO: sample n points from N(mu0, sigma^2)
    x[theta:] = # TODO: overwrite the post-change part with N(mu1, sigma^2)
    return x


x = generate_mean_shift(N, THETA, rng=rng)

fig, ax = plt.subplots(figsize=(9, 2.8), dpi=140)
ax.plot(x, color="#1B1B1E", lw=1.0)
ax.axvline(THETA, color="#2f6f6f", lw=1.6, label=r"$\theta$ (true change)")
ax.set_xlabel("time t")
ax.set_ylabel(r"$x_t$")
ax.set_title("Series with a mean shift")
ax.legend(frameon=False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(True, alpha=0.25, ls="--")
plt.tight_layout()
plt.show()


## 2. Derive the increment $\ell_t$ for the Gaussian case

Assume

$$
f_0 = \mathcal{N}(\mu_0, \sigma^2),
\qquad
f_1 = \mathcal{N}(\mu_1, \sigma^2),
$$

with known $\mu_0$, $\mu_1$, and $\sigma$.

Write down

$$
\ell_t = \log \dfrac{f_1(x_t)}{f_0(x_t)}
$$

and simplify it.

Hint: start from the Gaussian density

$$
f(x) = \frac{1}{\sqrt{2\pi}\,\sigma}\exp\Bigl(-\frac{(x-\mu)^2}{2\sigma^2}\Bigr)
$$

and see what remains after taking the log-ratio. Constants that do **not** depend on $x_t$ can be dropped carefully — but keep everything that involves $\mu_0$, $\mu_1$, or $\sigma$.

In [ ]:
# TODO: after deriving ell_t on paper, implement it here

def llr_mean_shift(x_t, mu0=MU0, mu1=MU1, sigma=SIGMA):
    """Log-likelihood ratio for a single observation."""
    raise NotImplementedError("Derive ell_t and put the formula here")


## 3. Assemble Page-CUSUM

From here it is mostly bookkeeping:

1. take the previous value $S_{t-1}$ (start from $0$);
2. add your $\ell_t$;
3. if the result goes negative, reset to zero;
4. if $S_t > h$, record the alarm time $\tau$.

In [ ]:
def page_cusum(x, h=H, mu0=MU0, mu1=MU1, sigma=SIGMA):
    """
    One-sided Page-CUSUM.

    Returns
    -------
    S   : array of CUSUM values over time
    tau : first alarm index, or None if the threshold is never crossed
    """
    n = len(x)
    tau = None
    S = # TODO: allocate / initialize the CUSUM statistic
    # TODO: CUSUM loop
    return S, tau


S, tau = page_cusum(x)
print(f"true change time theta = {THETA}")
print(f"alarm time         tau = {tau}")


## 4. Two metrics: delay and time to false alarm

### Detection delay
Look at a series that **does** contain a change.

- if the alarm comes *after* $\theta$: $\mathrm{delay} = \tau - \theta$ — that is your lag;
- if the alarm comes *before* $\theta$: that is a false alarm, so we do not report a delay;
- if there is no alarm at all: a miss.

### Time to false alarm (TFA)
Now take a quiet series *without* a change — the mean stays $\mu_0$ forever.  
TFA is the time until the first false alarm. Larger is better: the detector is more patient.

If no alarm appears before the end of a long series, we call the run **censored** (threshold too high / series too short).

In [ ]:
def detection_delay(theta, tau):
    """Summarize one run on a series with a change."""
    if tau is None:
        return {"outcome": "miss", "delay": None, "false_alarm_before_cp": False}
    if tau < theta:
        return {"outcome": "false_alarm", "delay": None, "false_alarm_before_cp": True}
    return {"outcome": "detect", "delay": int(tau - theta), "false_alarm_before_cp": False}


def time_to_false_alarm(n=N_FA, mu0=MU0, mu1=MU1, sigma=SIGMA, h=H, rng=None):
    """TFA on a series with no change."""
    rng = rng or np.random.default_rng()
    x_fa = rng.normal(mu0, sigma, size=n)
    _, tau_fa = page_cusum(x_fa, h=h, mu0=mu0, mu1=mu1, sigma=sigma)
    return {
        "tfa": tau_fa,                 # None = no alarm within horizon n
        "censored": tau_fa is None,
        "horizon": n,
    }


delay_info = detection_delay(THETA, tau)
tfa_info = time_to_false_alarm(rng=np.random.default_rng(SEED + 1))

print("Result on series with a change:", delay_info)
print("Result on series with no change:", tfa_info)


## 5. Look at the run: series, $S_t$, and the times $\theta$ and $\tau$

Top panel: the observations.  
Bottom panel: how CUSUM grows and where it hits the threshold $h$.  
If the detector fires after the change, we draw a **delay** arrow above the curve.

In [ ]:
def plot_cusum_run(x, S, theta, tau, h=H, title="Page-CUSUM"):
    t = np.arange(len(x))
    fig, axes = plt.subplots(
        2, 1, figsize=(9, 4.6), dpi=140, sharex=True,
        gridspec_kw={"hspace": 0.12},
    )

    ax = axes[0]
    ax.plot(t, x, color="#1B1B1E", lw=1.0)
    ax.axvline(theta, color="#2f6f6f", lw=1.5, label=r"$\theta$")
    if tau is not None:
        ax.axvline(tau, color="#C0392B", lw=1.4, ls="--", label=r"alarm $\tau$")
    ax.set_ylabel(r"$x_t$")
    ax.set_title(title, loc="left", fontsize=11)
    ax.legend(frameon=False, loc="upper left")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(True, alpha=0.25, ls="--")

    ax = axes[1]
    ax.plot(t, S, color="#1B1B1E", lw=1.1, label=r"$S_t$")
    ax.axhline(h, color="#AAC508", lw=1.6, label=rf"threshold $h={h:g}$")
    ax.axvline(theta, color="#2f6f6f", lw=1.5)
    if tau is not None:
        ax.axvline(tau, color="#C0392B", lw=1.4, ls="--")
        ax.scatter([tau], [S[tau]], color="#C0392B", s=35, zorder=5)
    ax.set_xlabel("time t")
    ax.set_ylabel(r"$S_t$")
    ax.legend(frameon=False, loc="upper left")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(True, alpha=0.25, ls="--")

    info = detection_delay(theta, tau)
    if info["outcome"] == "detect":
        ymin, ymax = ax.get_ylim()
        span = ymax - ymin
        # lift the axis so the delay mark sits above the curve
        ax.set_ylim(ymin, ymax + 0.28 * span)
        y = ymax + 0.10 * span
        ax.annotate(
            "",
            xy=(tau, y),
            xytext=(theta, y),
            arrowprops=dict(arrowstyle="<->", color="#AAC508", lw=1.6),
            zorder=6,
        )
        ax.text(
            (theta + tau) / 2,
            y + 0.05 * span,
            f"delay = {info['delay']}",
            ha="center",
            va="bottom",
            color="#5C6B04",
            fontsize=9,
            fontweight="bold",
            bbox=dict(facecolor="white", edgecolor="none", alpha=0.9, pad=1.2),
            zorder=7,
        )


    plt.show()
    return info


plot_cusum_run(x, S, THETA, tau)


## 6. Average over many runs (Monte Carlo)

A single random series is noisy: delay can look great today and worse tomorrow.  
So we repeat the experiment $M$ times and report average delay and TFA.

In [ ]:
def monte_carlo(M=200, h=H, seed=0):
    rng = np.random.default_rng(seed)
    delays = []
    fa_before = 0
    misses = 0
    tfas = []
    tfa_censored = 0

    for _ in range(M):
        # series WITH a change
        x_i = generate_mean_shift(N, THETA, rng=rng)
        _, tau_i = page_cusum(x_i, h=h)
        info = detection_delay(THETA, tau_i)
        if info["outcome"] == "detect":
            delays.append(info["delay"])
        elif info["outcome"] == "false_alarm":
            fa_before += 1
        else:
            misses += 1

        # series WITHOUT a change
        x_fa = rng.normal(MU0, SIGMA, size=N_FA)
        _, tau_fa = page_cusum(x_fa, h=h)
        if tau_fa is None:
            tfa_censored += 1
        else:
            tfas.append(tau_fa)

    return {
        "M": M,
        "h": h,
        "mean_delay": float(np.mean(delays)) if delays else None,
        "median_delay": float(np.median(delays)) if delays else None,
        "n_detect": len(delays),
        "n_false_alarm_before_cp": fa_before,
        "n_miss": misses,
        "mean_tfa": float(np.mean(tfas)) if tfas else None,
        "median_tfa": float(np.median(tfas)) if tfas else None,
        "n_tfa_observed": len(tfas),
        "n_tfa_censored": tfa_censored,
    }


stats = monte_carlo(M=200, h=H, seed=7)
for key, val in stats.items():
    print(f"{key:28s}: {val}")


## 7. Play with the threshold $h$

Compare a few values of $h$. The usual picture is:

- small $h$ → you catch the change faster, but cry wolf more often (TFA drops);
- large $h$ → fewer false alarms, but you react later.

That trade-off is the heart of online change-point detection.

In [ ]:
hs = # TODO: pick a few thresholds h to compare
rows = [monte_carlo(M=150, h=h, seed=11) for h in hs]

print(f"{'h':>6}  {'mean_delay':>12}  {'mean_tfa':>12}  {'FA_before':>10}  {'miss':>6}")
for r in rows:
    md_ = f"{r['mean_delay']:.2f}" if r["mean_delay"] is not None else "—"
    mt_ = f"{r['mean_tfa']:.1f}" if r["mean_tfa"] is not None else "censored"
    print(
        f"{r['h']:6.1f}  {md_:>12}  {mt_:>12}  "
        f"{r['n_false_alarm_before_cp']:10d}  {r['n_miss']:6d}"
    )


## 8. See the trade-off on a plot

Numbers in a table are fine, but the trade-off is easier to feel on a figure.

Using the thresholds from the previous cell, plot:

- mean detection delay vs $h$;
- mean time to false alarm vs $h$.

You can put both curves on one figure (two $y$-axes) or use two side-by-side panels.  
Ask yourself: as $h$ grows, which curve goes up, and why?

In [ ]:
# TODO: using hs and rows from the previous cell,
# plot mean delay and mean TFA against the threshold h.

fig, ax1 = plt.subplots(figsize=(7.2, 3.2), dpi=140)

delays = # TODO: list of mean_delay from rows
tfas = # TODO: list of mean_tfa from rows  (use np.nan if censored / None)

ax1.plot(hs, delays, "o-", color="#2f6f6f", lw=1.8, label="mean delay")
ax1.set_xlabel(r"threshold $h$")
ax1.set_ylabel("mean detection delay", color="#2f6f6f")
ax1.tick_params(axis="y", labelcolor="#2f6f6f")

ax2 = ax1.twinx()
# TODO: plot mean TFA on ax2 (suggestion: "s--" and color "#AAC508")
ax2.set_ylabel("mean time to false alarm", color="#5C6B04")
ax2.tick_params(axis="y", labelcolor="#5C6B04")

ax1.set_title("Delay vs TFA as the threshold changes", loc="left", fontsize=11)
ax1.grid(True, alpha=0.25, ls="--")
ax1.spines["top"].set_visible(False)

# TODO: add a combined legend for both curves

plt.tight_layout()
plt.show()


## What to show in the seminar

1. A plot: the series + $S_t$, with $\theta$ and $\tau$ marked.
2. Numbers: delay on one example, plus Monte Carlo averages for delay and TFA.
3. The delay / TFA figure against $h$, plus a short takeaway in your own words.